In [7]:
from datasets import load_dataset
from langchain_core.documents import Document

# dataset = load_dataset("ted_talks_iwslt", "nl_en_2016", split="train[:1000]")  # Limit for memory
dataset = load_dataset(
    "IWSLT/ted_talks_iwslt",
    language_pair=("nl", "en"),
    year="2016",
    split="train[:1000]",
    trust_remote_code=True
)

documents = []
for item in dataset:

    if item["translation"]["en"]:
        doc = Document(
            page_content=item["translation"]["en"],
            metadata={

                "speaker": item.get("speaker", "Unknown"),
                "translation_pair": "nl_en_2016"

            }
        )
documents.append(doc)


Generating train split:   0%|          | 0/3852 [00:00<?, ? examples/s]

In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# =====================================================================
# Split the document. Each chunk has 500 characters, and 50 characters overlap with the next chunk for context continuity.
# =====================================================================
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
docs = text_splitter.split_documents(documents[:50])  # limit size to reduce memory
print(docs)

[Document(metadata={'speaker': 'Unknown', 'translation_pair': 'nl_en_2016'}, page_content='Two young scientists break down plastics with bacteria')]


In [9]:
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

# =====================================================================
# Embed using Hugging Face sentence transformer
# =====================================================================
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# =====================================================================
# Store embeddings in FAISS
# =====================================================================

db = FAISS.from_documents(
    docs,
    embeddings
)
print(db)

/var/folders/mk/3k8t72sj69l04zsskj14q1j00000gn/T/ipykernel_10658/3034470447.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS
/var/folders/mk/3k8t72sj69l04zsskj14q1j00000gn/T/ipykernel_10658/3034470447.py:7: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [42]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
# =====================================================================
# Load FLAN-T5 directly
# =====================================================================
model_name = "google/flan-t5-small"
# model_name = "meta-llama/Llama-3.2-1B-Instruct" # we can try this as well whcih is good but need authentication

tokenizer = AutoTokenizer.from_pretrained(
    model_name
)

model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name, tie_word_embeddings=False
)

model.eval()

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] T5ForConditionalGeneration LOAD REPORT from: google/flan-t5-small
Key                         | Status  | 
----------------------------+---------+-
encoder.embed_tokens.weight | MISSING | 
decoder.embed_tokens.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=384, bias=False)
              (k): Linear(in_features=512, out_features=384, bias=False)
              (v): Linear(in_features=512, out_features=384, bias=False)
              (o): Linear(in_features=384, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 6)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseGatedActDense(
              (wi_0): Linear(in_features=512, out_features=1024, bias=False)
              (wi_1): Linear(in_features=512, out_features=1024, bias=False)
              (wo): 

In [43]:
from langchain_huggingface import HuggingFacePipeline
from transformers import pipeline

# =====================================================================
# Create Hugging Face pipeline
# =====================================================================

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=100,
)

# =====================================================================
# Wrap Hugging Face pipeline with LangChain
# =====================================================================

llm = HuggingFacePipeline(
    pipeline=pipe
)

# Test with example
response = llm.invoke(
    "Explain artificial intelligence in simple terms."
)

print(response)

[transformers] The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'AXK1ForCausalLM', 'AXK2ForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'Cohere2MoeForCausalLM', 'CohereCompassForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DeepseekV32ForCausalLM', 'DeepseekV4ForCausalLM', 'DiffLlamaForCausalLM', 'DogeForCausalLM', 'Dots1ForCausalLM', 'ElectraForCausalLM', 'Emu3ForCausalLM', 'ErnieForCausalLM',

Explain artificial intelligence in simple terms.


In [55]:
from langchain_classic.chains.retrieval import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains.history_aware_retriever import create_history_aware_retriever
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
# =====================================================================
# Build a chat prompt that explicitly lists context and chat history
# =====================================================================

chat_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a helpful chat assistant answering questions about TED Talks. "
        "Answer the user's latest question using only the provided context. "
        "If you do not know the answer, say 'I don't know'.\n\n"
        "Context:\n{context}"
    ),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}"),
])

# =====================================================================
# Set up a sub-prompt that re-phrases follow-up questions based on history
# =====================================================================

rephrase_prompt = ChatPromptTemplate.from_messages([
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}"),
    ("human", "Given the above conversation, generate a search query to look up context relevant to the conversation.")
])

# =====================================================================
# Convert your FAISS store into a history-aware retriever
# =====================================================================

retriever = db.as_retriever(search_kwargs={"k": 2})
history_aware_retriever = create_history_aware_retriever(
    llm, retriever, rephrase_prompt
)

# =====================================================================
# Create the document combining chain and the final conversational RAG chain
# =====================================================================

question_answer_chain = create_stuff_documents_chain(llm, chat_prompt)
conversational_rag_chain = create_retrieval_chain(history_aware_retriever, question_answer_chain)


In [54]:
from langchain_core.messages import HumanMessage, AIMessage

# =====================================================================
# Initialize an empty list to keep track of the chat logs
# =====================================================================

chat_history = []
# =====================================================================
# --- Turn 1 ---
# =====================================================================

query_1 = "What is the primary topic discussed in the first talk?"
response_1 = conversational_rag_chain.invoke({
    "input": query_1,
    "chat_history": chat_history
})
print(f"Human: {query_1}")
print(f"AI: {response_1['answer']}\n")

# Update history with the first interaction
chat_history.extend([
    HumanMessage(content=query_1),
    AIMessage(content=response_1["answer"])
])

# =====================================================================
# --- Turn 2 (Follow-up question relying on memory) ---
# =====================================================================
query_2 = "Can you give me more details about that?"
response_2 = conversational_rag_chain.invoke({
    "input": query_2,
    "chat_history": chat_history
})
print(f"Human: {query_2}")
print(f"AI: {response_2['answer']}")


[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Human: What is the primary topic discussed in the first talk?
AI: System: You are a helpful chat assistant answering questions about TED Talks. Answer the user's latest question using only the provided context. If you do not know the answer, say 'I don't know'.

Context:
Two young scientists break down plastics with bacteria
Human: What is the primary topic discussed in the first talk?aaaa ase andaa are are ands s and- for inloom t as to of cans



[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Human: Can you give me more details about that?
AI: System: You are a helpful chat assistant answering questions about TED Talks. Answer the user's latest question using only the provided context. If you do not know the answer, say 'I don't know'.

Context:
Two young scientists break down plastics with bacteria
Human: What is the primary topic discussed in the first talk?
AI: System: You are a helpful chat assistant answering questions about TED Talks. Answer the user's latest question using only the provided context. If you do not know the answer, say 'I don't know'.

Context:
Two young scientists break down plastics with bacteria
Human: What is the primary topic discussed in the first talk?aaaa ase andaa are are ands s and- for inloom t as to of cans
Human: Can you give me more details about that?
